# VVC_RL D-Series — Stage 1: Full Preflight
GPU T4 x2, Internet ON. Default D1, same variant and seed in all three notebooks. Functional commit c4bccd19f9d48ee2a8269b94d109d441044e4ff3. Add BDD100K only. No formal training. Save both named SAFE gate and TOOLCHAIN ZIP outputs. No Test access. A session timeout may require resume rather than finishing in exactly three sessions.

In [ ]:
STAGE = 1
from pathlib import Path
import sys, os, json, time, subprocess, shutil
SOURCE_COMMIT = 'c4bccd19f9d48ee2a8269b94d109d441044e4ff3'
REPOSITORY = 'https://github.com/youngan2810/VVC_RL.git'
VARIANTS = {
    'D0': ('encoder-feedback-v1','task-budget-v3-progressive-soft-iou'),
    'D1': ('d1-decoupled-v1','task-budget-v3-progressive-soft-iou'),
    'D1i': ('d1-isolated-v1','task-budget-v3-progressive-soft-iou'),
    'D2': ('d2-region-v1','task-budget-v3-progressive-soft-iou'),
    'D2-neutral': ('d2-region-neutral-v1','task-budget-v3-progressive-soft-iou'),
    'D3': ('d3-online-budget-v1','task-budget-v4-online-soft-iou'),
}
VARIANT = 'D1'
SEED = 20261120
FRAMES = 256
VAL_EVERY = 5
SESSION_HOURS = 11.0
# Optional: explicit mounted paths, if automatic discovery is ambiguous.
PREFLIGHT_INPUT = ''
TOOLCHAIN_INPUT = ''
TRAINING_INPUT = ''
# Only for continuing an interrupted stage; otherwise leave empty.
PARTIAL_STAGE_INPUT = ''
assert VARIANT in VARIANTS and 0 < SESSION_HOURS <= 11.75
STATE_VERSION, REWARD_VERSION = VARIANTS[VARIANT]
WORK=Path('/kaggle/working'); INPUT=Path('/kaggle/input')
assert WORK.is_dir() and INPUT.is_dir()
TAG=f'{VARIANT}_seed{SEED}_{SOURCE_COMMIT[:8]}'
BASE=WORK/('VVC_RL_D_'+SOURCE_COMMIT[:8]); REPO=BASE/'source'; TOOLS=BASE/'tools'
PREFLIGHT=WORK/f'preflight_{TAG}'
TRAIN=WORK/f'vvc_rl_{TAG}_stage{STAGE}'
GATE_NAME=f'VVC_RL_PREFLIGHT_{PREFLIGHT.name}_SAFE.zip'
TOOLS_NAME=f'VVC_RL_{TAG}_TOOLCHAIN.zip'
RUN10_NAME=f'VVC_RL_MINI_vvc_rl_{TAG}_stage2_SAFE.zip'
RUN40_NAME=f'VVC_RL_MINI_vvc_rl_{TAG}_stage3_SAFE.zip'
LOGS=WORK/f'logs_stage{STAGE}_{TAG}_{time.time_ns()}'; LOGS.mkdir(parents=True)
STARTED=time.monotonic()
os.environ['YOLOv5_AUTOINSTALL']='false'
print('STAGE',STAGE,VARIANT,STATE_VERSION,REWARD_VERSION)


In [ ]:
def live(args,name,cwd=None):
    args=list(map(str,args)); log=LOGS/f'{name}.log'
    print('$',*args,flush=True)
    with log.open('w',encoding='utf-8') as out:
        p=subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,errors='replace')
        try:
            for line in p.stdout:
                out.write(line);out.flush();print(line,end='',flush=True)
            if p.wait():raise RuntimeError(f'{name} failed; see {log}')
        finally:
            if p.poll() is None:p.terminate();p.wait()
    return log

def common_args():
    return ['--toolchain',TOOLS/'TOOLCHAIN.json','--tools',TOOLS,'--input',INPUT,
        '--frames',FRAMES,'--state-version',STATE_VERSION,'--reward-version',REWARD_VERSION,
        '--rate-tolerance','0.02','--gamma','1.0','--gae-lambda','0.95','--train-schedule','mixed','--seed',SEED]

def verify_source():
    script="""import sys,subprocess
from vvcrl.repository import verify_checkout
from vvcrl.config import fingerprint
assert subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()==sys.argv[1]
print('TRACKED_RAW_BYTES_PASS',verify_checkout('.'))
# No untracked functional input may silently change the source fingerprint.
assert not subprocess.check_output(['git','ls-files','--others','--exclude-standard','--','vvcrl','native','vendor','configs','scripts','tests'],text=True).strip()
print('FUNCTIONAL_SOURCE_SHA256',fingerprint('.'))
"""
    live([sys.executable,'-c',script,SOURCE_COMMIT],'source_verify',REPO)

def gate_check():
    script="""from pathlib import Path
import json,sys
from vvcrl.gates import binding,require_gate
from vvcrl.config import Profile,PPOConfig,RewardConfig
p,t,sv,rv,frames=sys.argv[1:]
b=binding(Path.cwd(),json.loads(Path(t).read_text()),Profile(frames=int(frames)),2,RewardConfig(rv,.02),PPOConfig(),'mixed',sv)
print('GATE_RECHECK_PASS',require_gate(p,b)['status'])
"""
    live([sys.executable,'-c',script,PREFLIGHT,TOOLS/'TOOLCHAIN.json',STATE_VERSION,REWARD_VERSION,FRAMES],'gate_recheck',REPO)

import zipfile, hashlib, tempfile
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def locate(name,explicit=''):
    if explicit:
        p=Path(explicit);assert p.is_file(),str(p)
        return p
    hits=sorted(p for p in INPUT.rglob(name) if p.is_file())
    if len(hits)==1:return hits[0]
    if len(hits)>1:raise RuntimeError(f'Ambiguous {name}: {hits}; set explicit input')
    # Some Kaggle notebook outputs expose only an outer _output_.zip.
    found=[]
    for wrapper in sorted(INPUT.rglob('_output_.zip')):
        with zipfile.ZipFile(wrapper) as z:
            for info in z.infolist():
                if not info.is_dir() and Path(info.filename).name==name:
                    found.append((wrapper,info.filename))
    if len(found)!=1:raise RuntimeError(f'Expected one {name}; found {found}. Add the correct previous notebook Output.')
    wrapper,member=found[0]
    folder=BASE/'mounted_packages';folder.mkdir(parents=True,exist_ok=True)
    p=folder/name;tmp=p.with_suffix('.partial')
    with zipfile.ZipFile(wrapper) as z,z.open(member) as src,tmp.open('wb') as dst:
        shutil.copyfileobj(src,dst,1024*1024)
    tmp.replace(p);return p

def import_safe(package,destination,label):
    assert not destination.exists() or not any(destination.iterdir()), f'Nonempty {destination}: use a fresh session'
    live([sys.executable,'-u','-m','vvcrl','import','--output',destination,'--package',package],label,REPO)

def save_toolchain():
    gate=json.loads((PREFLIGHT/'PREFLIGHT_GATE.json').read_text())
    tool=json.loads((TOOLS/'TOOLCHAIN.json').read_text())
    hashes=gate['binding']['binary_sha256']
    assert all(sha256(tool[k])==hashes[k] for k in ('encoder','vanilla','decoder'))
    target=WORK/TOOLS_NAME;tmp=target.with_suffix('.partial')
    with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3) as z:
        z.writestr('manifest.json',json.dumps({'source_commit':SOURCE_COMMIT,'binding':gate['binding']}))
        for k in ('encoder','vanilla','decoder'):z.write(tool[k],k)
    tmp.replace(target)
    print('TOOLCHAIN_READY',target)

def restore_toolchain(package):
    gate=json.loads((PREFLIGHT/'PREFLIGHT_GATE.json').read_text())
    TOOLS.mkdir(parents=True,exist_ok=True)
    binaries={}
    with zipfile.ZipFile(package) as z:
        assert sorted(z.namelist())==['decoder','encoder','manifest.json','vanilla']
        m=json.loads(z.read('manifest.json'))
        assert m['source_commit']==SOURCE_COMMIT and m['binding']==gate['binding']
        for k in ('encoder','vanilla','decoder'):
            p=TOOLS/k;tmp=p.with_suffix('.partial')
            with z.open(k) as src,tmp.open('wb') as dst:shutil.copyfileobj(src,dst,1024*1024)
            assert sha256(tmp)==gate['binding']['binary_sha256'][k],k
            tmp.chmod(0o755);tmp.replace(p);binaries[k]=str(p)
    (TOOLS/'TOOLCHAIN.json').write_text(json.dumps(binaries,indent=2))
    # Install the same dependency set as setup_kaggle.py, but NEVER rebuild the
    # native binaries. Restore pinned detector code and verified weights.
    live(['apt-get','update','-qq'],'apt_update')
    live(['apt-get','install','-y','-qq','cmake','g++','ffmpeg','pkg-config','libswscale-dev','libavutil-dev'],'apt_deps')
    live([sys.executable,'-m','pip','install','--disable-pip-version-check','setuptools==80.9.0','gitpython==3.1.45','requests','PyYAML','pandas','seaborn','scipy','pillow','tqdm','psutil','opencv-python-headless'],'detector_deps')
    script="""import sys,urllib.request
from pathlib import Path
sys.path.insert(0,str(Path.cwd()/'scripts'))
from setup_kaggle import pinned_repo
from vvcrl.data import DETECTOR_ID
from vvcrl.config import digest
tools=Path(sys.argv[1])
pinned_repo('https://github.com/ultralytics/yolov5.git',tools/'yolov5',DETECTOR_ID['source'])
w=tools/'yolov5n.pt'
if not w.exists():
    tmp=w.with_suffix('.download')
    urllib.request.urlretrieve('https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5n.pt',tmp)
    tmp.replace(w)
assert digest(w)==DETECTOR_ID['weights']
"""
    live([sys.executable,'-c',script,TOOLS],'detector_restore',REPO)
    verify_source();gate_check()

def show_progress():
    p=TRAIN/'PROGRESS.json'
    if not p.is_file():
        print('NO_CHECKPOINT_YET');return None
    r=json.loads(p.read_text());print('PROGRESS',json.dumps(r,indent=2))
    assert r['episodes']==2*r['iteration']
    assert sha256(TRAIN/r['checkpoint'])==r['checkpoint_sha256']
    for p in sorted((TRAIN/'validations').glob('*.json')):
        v=json.loads(p.read_text());print('VAL',p.stem,v['complete'],v['feasible'],v['macro_delta_quality'],v['macro_delta_utility'])
    return r


## Exact source and dataset

In [ ]:
BASE.mkdir(parents=True,exist_ok=True)
if not REPO.exists():live(['git','clone','--branch','research/d-series-adaptive-policy',REPOSITORY,REPO],'clone')
else:
    assert (REPO/'.git').is_dir()
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain','--untracked-files=no'],text=True).strip(),'Local tracked edits'
live(['git','-C',REPO,'fetch','--no-tags','origin',SOURCE_COMMIT],'fetch')
live(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT],'checkout')
verify_source()
live([sys.executable,'-c','import torch; print(torch.__version__,torch.cuda.device_count()); assert torch.cuda.is_available() and torch.cuda.device_count()>=2'], 'gpu_check',REPO)
live([sys.executable,'-c',"from vvcrl.locked.data import VideoCatalog; import sys; c=VideoCatalog(sys.argv[1],mode='mini',seed=20261111); c.check_available(('train','val')); print('DATASET_PASS',len(c.selected['train']),len(c.selected['val']))",INPUT],'dataset_check',REPO)
print('FREE_GB',shutil.disk_usage(WORK).free/1e9)

## Build and full preflight

In [ ]:
live([sys.executable,'-u',REPO/'scripts/setup_kaggle.py','--tools',TOOLS,'--jobs',2],'setup',REPO)
verify_source()
assert not PREFLIGHT.exists(),'Preflight folder exists: use a fresh session'
if not PREFLIGHT.exists():
    live([sys.executable,'-u','-m','vvcrl','preflight','--output',PREFLIGHT,*common_args()],'preflight',REPO)
gate_check()
gate=json.loads((PREFLIGHT/'PREFLIGHT_GATE.json').read_text())
assert gate['status']=='PASS_READY_FOR_MINI'
print(json.dumps(gate['checks'],indent=2))
stats=json.loads((PREFLIGHT/'PPO_MECHANICAL_UPDATE.json').read_text())['stats']
print('PPO_MECHANICS',json.dumps(stats,indent=2))
summary=json.loads((PREFLIGHT/'REAL_DATA_SUMMARY.json').read_text())
assert summary['test_used'] is False and summary['policy_changed_on_validation'] is False
for split in ('train','validation'):
    for r in summary[split]:
        assert r['reward_identity_error']<1e-10
        print(split,r['rate'],r['actual_kbps'],r['task_retention_f1_v2'],r['reward_identity_error'])
print('PREFLIGHT_SAFE',WORK/f'VVC_RL_PREFLIGHT_{PREFLIGHT.name}_SAFE.zip')
save_toolchain()
assert (WORK/GATE_NAME).is_file()
print('STAGE_STATUS COMPLETE')
print('GATE_PACKAGE',WORK/GATE_NAME)
print('TOOLCHAIN_PACKAGE',WORK/TOOLS_NAME)


## Output handoff
Save Version with the required files in Kaggle Output. A failed or partial run is not a completed stage. Keep the same source, environment, seed, variant and binary identities; never bypass a failed gate.